In [1]:
# Standard library
import warnings
from operator import itemgetter

# Third-party
import bs4
from dotenv import load_dotenv
from pydantic import BaseModel, Field, PydanticDeprecatedSince20

# LangChain - core
from langchain_core.load import dumps, loads
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_core.documents import Document

# LangChain - integrations
from langchain_chroma.vectorstores import Chroma
from langchain_community.document_loaders import WebBaseLoader
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

# LangSmith
from langsmith import Client

# Suppress Pydantic v2 deprecation warnings from LangChain internals
warnings.filterwarnings("ignore", category=PydanticDeprecatedSince20)

load_dotenv()

/tmp/ipykernel_7995/3989172771.py:19: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader


True

In [2]:
from openai import OpenAI  

client = OpenAI()  

response = client.responses.create( 
    model="openai.gpt-oss-120b", 
    input=[ 
        {"role": "user", "content": "Write a one-sentence bedtime story about a unicorn."} 
    ] 
)  

print(response.output_text)

Under the silver glow of a sleepy moon, a gentle unicorn tiptoed across a meadow of whispering lullaby‑lilies, its soft horn sprinkling starlight dust that turned the night’s sighs into sweet dreams for every child tucked snugly in their blankets.


In [3]:
%pip install boto3

Note: you may need to restart the kernel to use updated packages.


In [4]:
# run in terminla
# aws login

In [5]:
import boto3

client = boto3.client('bedrock-agent-runtime')

In [6]:
# Install boto3 if not already installed (uncomment the line below)
# !pip install boto3

import boto3

# ========== CONFIGURE THESE THREE VALUES ==========
KNOWLEDGE_BASE_ID = "ZTH6KWCY0T"  # Your Knowledge Base ID
MODEL_ARN = "arn:aws:bedrock:us-east-1:279566174663:inference-profile/global.anthropic.claude-haiku-4-5-20251001-v1:0"
REGION = "us-east-1"  # Same region as your Knowledge Base
# ==================================================

# Ask your question
question = "What is agent memory?"

# Create client
client = boto3.client('bedrock-agent-runtime', region_name=REGION)

# Call the API
response = client.retrieve_and_generate(
    input={'text': question},
    retrieveAndGenerateConfiguration={
        'type': 'KNOWLEDGE_BASE',
        'knowledgeBaseConfiguration': {
            'knowledgeBaseId': KNOWLEDGE_BASE_ID,
            'modelArn': MODEL_ARN
        }
    }
)

# Print the answer
print("Answer:", response['output']['text'])

# Optional: print source documents
print("\nSources:")
for citation in response.get('citations', []):
    for ref in citation.get('retrievedReferences', []):
        uri = ref.get('location', {}).get('s3Location', {}).get('uri', 'No URI')
        print(" -", uri)

Answer: Agent memory is a component that enables large language models (LLMs) to function as autonomous agents. While LLMs are not autonomous agents by themselves, memory can be integrated as a tool or provided as additional input to give them the ability to retain and use information over time.

In practice, agent memory works by storing information from past interactions or episodes. For example, in the Reflexion method, after each episode completes, the LLM is given a record of what happened and prompted to extract "lessons learned" from that experience. These lessons are then stored as long-term memory and provided to the agent in subsequent episodes, allowing it to improve its performance based on past experiences.

This capability transforms a basic LLM into an agent that can recall past behaviors, plan future actions, and interact with dynamic environments more effectively.

Sources:
 - s3://first-ml-bucket-2000/Large language model.txt
